In [1]:
# import sys
# sys.path.append("../")
import os

os.chdir("/archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/dev2/scMEDAL_for_scRNAseq")
from models.models import train_model_on_named_experiment

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


## Training Models:

#### Implemented Models:
| Model Class | Model Description | 
|------------|--------------------|  
| `AE` | Autoencoder | 
| `AEC`| Autoencoder Classifier |
| `scMEDAL-FE` | Domain Adversarial Autoencoder |
| `scMEDAL-FEC`| Domain Adversarial Autoencoder Classifier |
| `scMEDAL-RE`| Domain Enhancing Autoencoder Classifier | 

#### Named Experiments:
| Valid Named Experiment | Dataset |  n_clusters | n_pred |
|------------------------|---------|-------------|--------|
| `AML`| Acute Myeloid Leukemia | 19 | 21 |
| `ASD`| Autism Spectrum Disorder | 31 | 17 | 
| `HH` | Healthy Heart | 147 | 13 | 

**Note:** If training on other datasets, the configs will need to be passed in as dictionaries to `model_kwargs` and `train_kwargs`.

`quick` is a boolean flag that can be passed to `train_kwargs` which shortens training to only 1 fold for 3 epochs.

In [ ]:
from utils.defaults import HH_OUTPUTS_DIR
import os
model_folder_dict = {
    #"ae":"",
    #"ae":"",
    #"aec":"",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-18",
    #"scmedalfec":"",
    "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-59",
}
model_folder_dict = {k:os.path.join(HH_OUTPUTS_DIR, "latent_space", "log_transformed_3000hvggenes",k, v) for k, v in model_folder_dict.items()}
print(model_folder_dict)
mec_asd = train_model_on_named_experiment("MEC", "HH", 
                                    train_kwargs={"quick":True, "results_path_dict":model_folder_dict, },
                                    model_kwargs={"models_list":list(model_folder_dict.keys()), #"get_pca":True,
                                                    #"latent_keys_config":{"fe_latent":"scmedalfe_latent", "re_latent":"scmedalre_latent"}}
                                                    "latent_keys_config":{"fe_latent":"scmedalfe_latent", "re_latent":"scmedalre_latent"}})
                                                # "latent_keys_config":{"fe_latent":"X_pca", "re_latent":"re_latent"}}
                                

In [ ]:
model_folder_dict

## Analyzing Models:

In [3]:
from analysis.analysis import HHAnalysis as hh
model_folder_dict = {
    #"ae":"",
    #"ae":"",
    #"aec":"",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-18",
    #"scmedalfec":"",
    "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-59",
}
# build the analysis object once
hh = hh(
    model_result_folder_dict = model_folder_dict,
    analysis_name            = "HH_default"     # optional tag for output folders
)


res= hh.clustering_scores(model_folder_dict,dataset_type="test")
# res

Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/HH/compare_models/log_transformed_3000hvggenes/HH_default
scmedalfe /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/HH/latent_space/log_transformed_3000hvggenes/scmedalfe/run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-18
scmedalre /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/HH/latent_space/log_transformed_3000hvggenes/scmedalre/run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-08_15-59


Mean scores paths:
  sample_si

In [ ]:


batches = ["H0037_Apex", "HCAHeart7836681", "HCAHeart8102861", "H0015_septum"]
# for this genomap config, you need to compute first the reconstructions from scMEDAL-FE and scMEDAL-RE
gfd = hh.genomap(model_folder_dict,
                            n_batches = 147,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "_index",
                            gene_index_col = "_index",
                            celltype = ["Ventricular_Cardiomyocyte", "Endothelial", "Fibroblast", "Pericytes"],
                            batches=batches,
                            models=['scmedalre'],#if add_inputs_fe=True-> scmedalfe+ inputs are used for genomap creation by default, no need to add the to the list, 
                            types=["train"], 
                            splits=[1],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            min_val = -2,
                            max_val = 8)#,issparse=True )

In [4]:
processors = hh.umap(model_folder_dict, types=["train"], splits=[1])
processors

\plot configs dict_items([('markers', ['x', '+', '<', 'h', 's', '.', 'o', 's', '^', '*', '1', '8', 'p', 'P', 'D', '|', 0, ',', 'd', 2]), ('palette_choice', ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6', '#000000', '#fabebe', '#008080', '#e6beff', '#9a6324', '#d2f53c', '#7fff00', '#000080', '#800000', '#808000', '#800080', '#808080', '#ffd700', '#ff4500', '#00ff7f', '#dda0dd', '#87ceeb', '#00ced1', '#9400d3', '#ff6347', '#4682b4', '#6b8e23', '#a52a2a']), ('showplot', False), ('save_fig', True), ('outpath', None), ('use_bio_bio', True), ('use_batch_batch', True), ('use_bio_batch', False), ('use_batch_bio', False)])
{'scmedalfe': '/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev2/scMEDAL_for_scRNAseq/outputs/HH/latent_space/log_transformed_3000hvggenes/scmedalfe/run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_b

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Sampling 19 batches out of 147
Computing UMAP on 41544 cells × 3000 features, using 'X_pca'
UMAP shape: (41544, 2)
Plotting input latent space.. for 19


/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Plotting scmedalfe_latent_train_1 latent space.. for 19
Computing UMAP on 41544 cells × 50 features, using 'X'
UMAP shape: (41544, 2)


/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Plotting scmedalre_latent_train_1 latent space.. for 19
Computing UMAP on 41544 cells × 50 features, using 'X'
UMAP shape: (41544, 2)
